# Unit 01 — Vectors & Vector Spaces

**Module 00: Orientation & Prerequisites** | fischer³ Education

| | |
|---|---|
| **Estimated time** | 50–60 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridge** | Ridge regression penalty |

---

## Learning Objectives

By the end of this unit you will be able to:

- Represent points and directions as vectors in $\mathbb{R}^n$ and state the difference between them
- Perform vector addition and scalar multiplication, algebraically and geometrically
- Compute the Euclidean norm of a vector and normalize it to a unit vector
- Express any vector as a linear combination of basis vectors
- Recognize vectors as the fundamental data structure of statistical models

## Lab Setup
Lab setup to run on [Google's Colab](https://colab.research.google.com) and elsewhere

In [ ]:
# ============================================================
# Install required libraries (safe to run in Colab or locally)
# ============================================================
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

required = [
    "numpy",
    "matplotlib",
    "sympy",
    "scipy",
]

for package in required:
    install(package)

print("All packages installed.")

In [ ]:
# ============================================================
# Imports and plot configuration
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 2,
    'figure.dpi': 120,
})

# Course colour palette
TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print("Setup complete.")

---

## 1. What Is a Vector?

A vector is one of two things, depending on context — and keeping that distinction clear will matter throughout this course.

**As a point**: a vector $\boldsymbol{x} \in \mathbb{R}^n$ specifies a location in $n$-dimensional space — the endpoint of an arrow drawn from the origin.

**As a direction**: a vector $\boldsymbol{x} \in \mathbb{R}^n$ specifies a magnitude and direction of travel, with no fixed starting position.

Both interpretations appear constantly in statistics and machine learning. A data point (a single observation with $n$ measured features) is a *point* in $\mathbb{R}^n$. The gradient of a loss function is a *direction* — the direction of steepest increase — with no fixed location.

**Definition.** The set $\mathbb{R}^n$ is the collection of all ordered $n$-tuples of real numbers:

$$
\mathbb{R}^n = \left\{ \boldsymbol{x} = (x_1, x_2, \ldots, x_n) \mid x_i \in \mathbb{R} \text{ for all } i \right\}
$$

Each element $\boldsymbol{x} \in \mathbb{R}^n$ is a **vector**. The numbers $x_1, \ldots, x_n$ are its **components**. We write vectors as column vectors throughout:

$$
\boldsymbol{x} = \begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix}
$$

> **Notation.** This course uses bold lowercase for vectors ($\boldsymbol{x}, \boldsymbol{y}, \boldsymbol{\theta}$) and bold uppercase for matrices ($\boldsymbol{X}, \boldsymbol{\Sigma}$). Scalars are unbolded ($x, \theta, \lambda$).

---

## 2. Vector Operations

Two operations define the algebraic structure of $\mathbb{R}^n$.

**Vector addition.** For $\boldsymbol{x}, \boldsymbol{y} \in \mathbb{R}^n$:

$$
\boldsymbol{x} + \boldsymbol{y} = \begin{pmatrix} x_1 + y_1 \\ \vdots \\ x_n + y_n \end{pmatrix}
$$

Componentwise addition. Geometrically: the parallelogram rule — place the tail of $\boldsymbol{y}$ at the tip of $\boldsymbol{x}$; the sum is the vector reaching the far corner.

**Scalar multiplication.** For $c \in \mathbb{R}$, $\boldsymbol{x} \in \mathbb{R}^n$:

$$
c\boldsymbol{x} = \begin{pmatrix} cx_1 \\ \vdots \\ cx_n \end{pmatrix}
$$

Scales the magnitude by $|c|$. Reverses direction if $c < 0$. Collapses to the origin if $c = 0$.

**Worked example.** Let $\boldsymbol{a} = (2, -3, 1)^{\top}$ and $\boldsymbol{b} = (-1, 4, 2)^{\top}$:

$$
\boldsymbol{a} + \boldsymbol{b} = (1, 1, 3)^{\top}
\qquad
3\boldsymbol{a} - 2\boldsymbol{b} = (8, -17, -1)^{\top}
$$

---

## 3. The Euclidean Norm

The **Euclidean norm** (length) of $\boldsymbol{x} \in \mathbb{R}^n$ is:

$$
\|\boldsymbol{x}\| = \sqrt{x_1^2 + x_2^2 + \cdots + x_n^2} = \sqrt{\boldsymbol{x}^{\top} \boldsymbol{x}}
$$

The **unit vector** in the direction of $\boldsymbol{x$ is obtained by dividing by its norm:

$$
\hat{\boldsymbol{x}} = \frac{\boldsymbol{x}}{\|\boldsymbol{x}\|}, \qquad \|\hat{\boldsymbol{x}}\| = 1
$$

**Worked example.** For $\boldsymbol{a} = (2, -3, 1)^{\top}$:

$$
\|\boldsymbol{a}\| = \sqrt{4 + 9 + 1} = \sqrt{14} \approx 3.742
$$

$$
\hat{\boldsymbol{a}} = \frac{1}{\sqrt{14}}\begin{pmatrix}2\\-3\\1\end{pmatrix}
$$

> **Watch out.** The point $(3, -1, 2)$ and the vector $(3, -1, 2)^{\top}$ have the same numerical components but different geometric meanings. In statistics, *observations are points*; *gradients, residuals, and parameter updates are vectors*.

---

## 4. Linear Combinations & Basis Vectors

A **linear combination** of vectors $\boldsymbol{v}_1, \ldots, \boldsymbol{v}_k \in \mathbb{R}^n$ with scalars $c_1, \ldots, c_k \in \mathbb{R}$ is:

$$
c_1 \boldsymbol{v}_1 + c_2 \boldsymbol{v}_2 + \cdots + c_k \boldsymbol{v}_k = \sum_{i=1}^k c_i \boldsymbol{v}_i
$$

The **standard basis** of $\mathbb{R}^n$ is $\{\boldsymbol{e}_1, \ldots, \boldsymbol{e}_n\}$, where $\boldsymbol{e}_i$ has a $1$ in position $i$ and $0$s elsewhere. Every vector decomposes uniquely:

$$
\boldsymbol{x} = x_1 \boldsymbol{e}_1 + x_2 \boldsymbol{e}_2 + \cdots + x_n \boldsymbol{e}_n
$$

The components *are* the coefficients in the standard basis expansion — this always holds.

---

## 5. Statistical Bridge — Ridge Regression [GLM]

Vectors are the native language of statistical modeling. A **data matrix** $\boldsymbol{X} \in \mathbb{R}^{n \times p}$ contains $n$ observations, each a point in $\mathbb{R}^p$. A **parameter vector** $\boldsymbol{\beta} \in \mathbb{R}^p$ collects all model unknowns into a single geometric object.

The norm appears directly in **Ridge regression**, where we minimize:

$$
J(\boldsymbol{\beta}) = \|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|^2
$$

The penalty term $\lambda\|\boldsymbol{\beta}\|^2$ measures the *size* of the parameter vector and shrinks it toward the origin. The scalar $\lambda > 0$ controls how strongly.

This is not a statistical trick — it is a direct consequence of the geometry of $\mathbb{R}^p$. We will return to this in Module 04 (optimization) when we derive the Ridge solution analytically.

The Python section below visualizes this penalty surface.

---

## Python: Visualization & Verification

The cells below demonstrate and visualize the concepts above. **Run them in order.** Modify the parameters and re-run — that is where the learning happens.

To run this notebook interactively, click the **rocket icon** at the top of the page and select **Open in Colab**.

### Section 1 — Symbolic Verification

Verify the worked example from Section 2 using SymPy exact arithmetic. If your hand answer disagrees with SymPy, trust SymPy and find the error.

In [ ]:
a = sp.Matrix([2, -3, 1])
b = sp.Matrix([-1, 4, 2])

print(f'a                   = {a.T}')
print(f'b                   = {b.T}')
print(f'a + b               = {(a + b).T}')
print(f'3a - 2b             = {(3*a - 2*b).T}')
print(f'||a||               = {sp.sqrt(a.dot(a))}  =  {float(sp.sqrt(a.dot(a))):.6f}')
print(f'unit vector a-hat   = {(a / sp.sqrt(a.dot(a))).T}')

### Section 2 — Geometric Visualization: Vector Addition & Normalization

In [ ]:
def draw_vector(ax, origin, vec, color, label, lw=2.0, alpha=1.0):
    """Draw a labelled arrow from origin to origin + vec."""
    ax.annotate(
        '', xy=origin + vec, xytext=origin,
        arrowprops=dict(arrowstyle='->', color=color, lw=lw, alpha=alpha)
    )
    if label:
        mid = origin + vec * 0.6
        ax.text(mid[0] + 0.1, mid[1] + 0.1, label,
                color=color, fontsize=12, fontweight='bold')

# Work in R^2 using the first two components of a and b
a2 = np.array([2., -3.])
b2 = np.array([-1.,  4.])
o  = np.array([0.,  0.])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left panel: parallelogram rule ────────────────────────────────────────
ax = axes[0]
draw_vector(ax, o,  a2,      TS_BLUE,  r'$\mathbf{a}$')
draw_vector(ax, o,  b2,      TS_GREEN, r'$\mathbf{b}$')
draw_vector(ax, o,  a2 + b2, TS_RED,   r'$\mathbf{a}+\mathbf{b}$', lw=2.5)
draw_vector(ax, a2, b2,      TS_GREEN, '', lw=1.0, alpha=0.35)
draw_vector(ax, b2, a2,      TS_BLUE,  '', lw=1.0, alpha=0.35)
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.axhline(0, color=TS_GRAY, lw=0.8)
ax.axvline(0, color=TS_GRAY, lw=0.8)
ax.set_aspect('equal')
ax.set_title('Vector Addition: Parallelogram Rule', color=TS_BLUE, fontweight='bold')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# ── Right panel: normalization ─────────────────────────────────────────────
ax2 = axes[1]
theta_c = np.linspace(0, 2 * np.pi, 300)
ax2.plot(np.cos(theta_c), np.sin(theta_c),
         color=TS_GRAY, lw=1.2, ls='--', label='Unit circle')

test_vecs   = [a2, b2, np.array([3., 1.]), np.array([-2., -2.])]
vec_colors  = [TS_BLUE, TS_GREEN, TS_AMBER, TS_RED]
vec_labels  = [r'$\hat{\mathbf{a}}$', r'$\hat{\mathbf{b}}$',
               r'$\widehat{(3,1)}$', r'$\widehat{(-2,-2)}$']

for v, col, lab in zip(test_vecs, vec_colors, vec_labels):
    v_hat = v / np.linalg.norm(v)
    ax2.annotate('', xy=v_hat, xytext=o,
                 arrowprops=dict(arrowstyle='->', color=col, lw=2.0))
    ax2.plot(*v_hat, 'o', color=col, ms=8, label=lab)

ax2.set_xlim(-1.6, 1.6); ax2.set_ylim(-1.6, 1.6)
ax2.axhline(0, color=TS_GRAY, lw=0.8)
ax2.axvline(0, color=TS_GRAY, lw=0.8)
ax2.set_aspect('equal')
ax2.set_title('Normalization: All Directions Land on the Unit Circle',
              color=TS_BLUE, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_xlabel('$x_1$'); ax2.set_ylabel('$x_2$')

plt.suptitle('Module 00, Unit 01 — Vectors in $\\mathbb{R}^2$',
             fontsize=13, color=TS_GRAY)
plt.tight_layout()
plt.show()

**What do you see?**

1. The dashed lines in the left panel complete the parallelogram. Both paths — travelling $\boldsymbol{a}$ then $\boldsymbol{b}$, or $\boldsymbol{b}$ then $\boldsymbol{a}$ — reach the same endpoint. This is vector addition commutativity made geometric.
2. In the right panel, four vectors of very different lengths all produce unit vectors that land exactly on the unit circle. Normalization strips magnitude and keeps only direction.

### Section 3 — Unit Balls: $\ell_1$, $\ell_2$, $\ell_\infty$

The Euclidean norm is one member of a family of $\ell_p$ norms:

$$
\|\boldsymbol{x}\|_p = \left( \sum_{i=1}^n |x_i|^p \right)^{1/p}
$$

The **unit ball** $\{\boldsymbol{x} : \|\boldsymbol{x}\|_p \leq 1\}$ has a characteristic shape for each $p$. These shapes determine the geometry of regularization in machine learning — a connection we will revisit formally in Module 04.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    (1,   TS_AMBER, r'$\ell_1$ ball  (Lasso)',
     r'$|x_1| + |x_2| \leq 1$',
     'Diamond shape — corners on axes → sparse solutions'),
    (2,   TS_BLUE,  r'$\ell_2$ ball  (Ridge)',
     r'$\|\boldsymbol{x}\|_2 \leq 1$',
     'Circle — smooth shrinkage, no sparsity'),
    (100, TS_GREEN, r'$\ell_\infty$ ball',
     r'$\max_i |x_i| \leq 1$',
     'Square — limits only the largest component'),
]

for ax, (p, col, title, formula, insight) in zip(axes, configs):
    # Parameterise the boundary |x|^p + |y|^p = 1
    x = np.sign(np.cos(theta)) * np.abs(np.cos(theta)) ** (2 / p)
    y = np.sign(np.sin(theta)) * np.abs(np.sin(theta)) ** (2 / p)
    ax.fill(x, y, alpha=0.15, color=col)
    ax.plot(x, y, color=col, lw=2.5)
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.axhline(0, color=TS_GRAY, lw=0.6)
    ax.axvline(0, color=TS_GRAY, lw=0.6)
    ax.set_title(title, color=col, fontweight='bold')
    ax.set_xlabel(formula, fontsize=9)
    ax.text(0, -1.32, insight, ha='center', fontsize=8.5,
            color=TS_GRAY, style='italic')

plt.suptitle(
    'Unit Balls in $\\mathbb{R}^2$ — The Geometry of Regularization',
    fontsize=13, color=TS_GRAY
)
plt.tight_layout()
plt.show()

**What do you see?**

The $\ell_1$ ball has corners sitting on the coordinate axes. When a loss function contour first contacts this ball during optimization it almost always hits a corner — a point where one component is exactly zero. This is the geometric reason Lasso regression produces *sparse* solutions. The smooth $\ell_2$ ball has no corners, so contact can happen anywhere — no sparsity. We will derive this rigorously in Module 04.

### Section 4 — Statistical Bridge: The Ridge Penalty Surface

In [ ]:
beta1 = np.linspace(-3, 3, 200)
beta2 = np.linspace(-3, 3, 200)
B1, B2  = np.meshgrid(beta1, beta2)
penalty = B1**2 + B2**2   # ||beta||^2

fig = plt.figure(figsize=(15, 5))

# ── Left: 3-D surface ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(B1, B2, penalty, cmap='Blues', alpha=0.85, linewidth=0)
ax1.set_xlabel(r'$\beta_1$')
ax1.set_ylabel(r'$\beta_2$')
ax1.set_zlabel(r'$\|\boldsymbol{\beta}\|^2$')
ax1.set_title(r'Ridge Penalty $\|\boldsymbol{\beta}\|^2$', color=TS_BLUE)

# ── Middle: contour plot ───────────────────────────────────────────────────
ax2 = fig.add_subplot(132)
cs = ax2.contourf(B1, B2, penalty, levels=20, cmap='Blues')
plt.colorbar(cs, ax=ax2)
ax2.set_xlabel(r'$\beta_1$')
ax2.set_ylabel(r'$\beta_2$')
ax2.set_title('Contours of Penalty (Level Curves)', color=TS_BLUE)
ax2.set_aspect('equal')

# ── Right: Ridge path — how beta-hat shrinks as lambda grows ──────────────
rng = np.random.default_rng(42)
n, p = 50, 2
X = rng.standard_normal((n, p))
beta_true = np.array([2.0, -1.5])
y = X @ beta_true + 0.5 * rng.standard_normal(n)

lambdas     = np.logspace(-2, 3, 100)
ridge_betas = np.array(
    [np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)
     for lam in lambdas]
)

ax3 = fig.add_subplot(133)
ax3.semilogx(lambdas, ridge_betas[:, 0], color=TS_BLUE,
             lw=2, label=r'$\hat{\beta}_1$')
ax3.semilogx(lambdas, ridge_betas[:, 1], color=TS_AMBER,
             lw=2, label=r'$\hat{\beta}_2$')
ax3.axhline(0, color=TS_GRAY, lw=0.8, ls='--')
ax3.axhline(beta_true[0], color=TS_BLUE,  lw=1, ls=':', alpha=0.5)
ax3.axhline(beta_true[1], color=TS_AMBER, lw=1, ls=':', alpha=0.5)
ax3.set_xlabel(r'$\lambda$ (regularization strength)')
ax3.set_ylabel(r'$\hat{\boldsymbol{\beta}}$ (Ridge estimate)')
ax3.set_title(
    r'Ridge Path: $\hat{\boldsymbol{\beta}}(\lambda)$', color=TS_BLUE
)
ax3.legend()

plt.suptitle(
    'Statistical Bridge — Ridge Regression Geometry',
    fontsize=13, color=TS_GRAY
)
plt.tight_layout()
plt.show()

print(f'True beta:           {beta_true}')
print(f'OLS  (lambda -> 0):  {ridge_betas[0].round(3)}')
print(f'Ridge (lambda = 10): {ridge_betas[np.searchsorted(lambdas, 10)].round(3)}')

**What do you see?**

1. The penalty surface is a paraboloid — a bowl centred at the origin. Regularization pulls the optimizer toward that bowl.
2. The contours are concentric circles, confirming the $\ell_2$ geometry from Section 3.
3. As $\lambda$ increases, both coefficients shrink continuously toward zero (the dotted lines show the true values). Unlike Lasso, neither coefficient reaches exactly zero — the smooth $\ell_2$ geometry prevents this.

---

## Summary

| Concept | Formula |
|---|---|
| Vector in $\mathbb{R}^n$ | Ordered $n$-tuple, interpreted as point or direction |
| Vector addition | $(x_1 + y_1, \ldots, x_n + y_n)^{\top}$ — componentwise |
| Scalar multiplication | $(cx_1, \ldots, cx_n)^{\top}$ — scales magnitude, reverses if $c < 0$ |
| Euclidean norm | $\|\boldsymbol{x}\| = \sqrt{\boldsymbol{x}^{\top} \boldsymbol{x}}$ |
| Unit vector | $\hat{\boldsymbol{x}} = \boldsymbol{x} / \|\boldsymbol{x}\|$, satisfies $\|\hat{\boldsymbol{x}}\| = 1$ |
| Linear combination | $\sum_{i=1}^k c_i \boldsymbol{v}_i$ |
| Ridge penalty | $\lambda \|\boldsymbol{\beta}\|^2$ — shrinks $\boldsymbol{\beta}$ toward the origin |

**Up next:** Unit 02 — Dot Products & Cross Products

The dot product gives us a way to measure *alignment* between two vectors — the geometric foundation of projection, orthogonality, and least squares regression.

---
*Module 00, Unit 01 — Threat Surfaces | fischer³ Education*